# バンディット問題勉強用ノートブック
このノートブックは、1状態のマルコフ決定過程（MDP）と形式的に等価なバンディット問題を説明するために作成されました。バンディット問題は、有限のアクション集合 $\mathcal{A}$ と、現在のアクションに対する報酬の確率分布 $p$ のタプル $(\mathcal{A}, p)$ で定義されます。以下の import 文は必要なモジュールやパッケージを読み込むためのもので、最初は意味が分からなくても大丈夫です。必要ならば Python の本を参照してください。

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interact, fixed
%matplotlib inline

## バンディット問題の例
5台のスロットマシンがあり、エージェントは5つの離散的な行動を選択できます。エージェントはアクションの選択に応じてスカラー報酬を受け取ります。目標は、期待報酬を最大化する行動を見つけることです。

**注意点**

- `GaussianBandit` は教育用のシンプルなクラスです（`gymnasium.Env` のサブクラスではありません）。
- すべての確率的なルーチンは `seed` 引数を受け付け、複数シードの平均をとるための `run_experiment` を提供しています。

In [ ]:
class GaussianBandit:
    """Simple multi-armed Gaussian bandit for educational use.

    The parameters mu and sigma are randomly generated based on the initial seed
    to provide different experiment setups for different students.
    """

    def __init__(self, seed=None):
        # Use a temporary RNG to generate the environment parameters
        param_rng = np.random.default_rng(seed)
        self.n_actions = 5
        # Randomly generate mu between 0 and 10, and sigma between 0.5 and 2.0
        self.mu = param_rng.uniform(0.0, 10.0, size=self.n_actions)
        self.sigma = param_rng.uniform(0.5, 2.0, size=self.n_actions)

        self.reset(seed=seed)

    # --- core API ---
    def reset(self, seed=None):
        # This RNG is for the stochastic rewards during the episode
        self.rng = np.random.default_rng(seed)
        return None

    def step(self, action):
        """Return a scalar reward drawn from N(mu[action], sigma[action])."""
        return float(self.rng.normal(self.mu[action], self.sigma[action]))

    # --- utilities ---
    @property
    def optimal_action(self):
        return int(np.argmax(self.mu))

    def report_settings(self):
        """Print the environment settings for a report."""
        print("--- Bandit Environment Report ---")
        print(f"Number of Actions: {self.n_actions}")
        for i in range(self.n_actions):
            print(f"Action {i}: mean(̐) = {self.mu[i]:.4f}, std(̓) = {self.sigma[i]:.4f}")
        print(f"Optimal Action: {self.optimal_action}")
        print("---------------------------------")

    def plot(self):
        fig = plt.figure(figsize=(8, 5))
        axarr = fig.subplots(1, 1)
        x = np.linspace(np.min(self.mu) - 4, np.max(self.mu) + 4, 500)
        for index, mean in enumerate(self.mu):
            std = self.sigma[index]
            norm_pdf = stats.norm.pdf(x=x, loc=mean, scale=std)
            axarr.plot(x, norm_pdf, label='$a%i$' % index)
        axarr.legend()
        axarr.set_xlabel('Reward (r)')
        axarr.set_ylabel('Probability Density')
        axarr.set_title('Reward Distributions per Action')
        axarr.grid()
        plt.show()

## 報酬を与える確率分布の可視化
この例題ではガウス分布で報酬を確率的に生成しています。
先頭の`seed=2026`となっている箇所の数値 2026 を学籍番号の数値と置き換えてからセルを実行してください。

レポートには`env.report_settings()`で表示される5個のガウス分布の平均と標準偏差を明記してください。

In [ ]:
env = GaussianBandit(seed=2026) # 学生ごとに異なるシード値、たとえば学籍番号の数字部分を設定してください
env.report_settings()
env.plot()

## 一様ランダム方策

より洗練されたアクション選択手法を検討する前に、シンプルな一様ランダム方策でベースラインを確認しましょう。この手法では、エージェントは完全にランダムにアクションを選択します。つまり、利用可能な各アクションが等しい確率で選択されます。各アクションのQ値（期待報酬）は、そのアクションが選択されたときに受け取ったすべての報酬の算術平均として推定されます。

In [ ]:
def uniform_random_method(env, number_of_steps=1000, seed=None, show_plot=True):
    """Single-run bandit experiment using a uniform random policy.

    Parameters
    ----------
    number_of_steps : int
    seed : int or None
        Random seed that controls both the environment and action sampling.
    show_plot : bool
        If True, draw the per-step Q and N curves.

    Returns
    -------
    dict with keys:
        Q : (n_actions, T+1) array
        N : (n_actions, T+1) array
        rewards : (T,) array
        actions : (T,) int array
        optimal_action : int
    """
    env.report_settings()
    rng = np.random.default_rng(None if seed is None else seed + 10_000)

    n_actions = env.n_actions

    # Initialize Q and N to store trajectories
    Q_trajectory = np.zeros((n_actions, number_of_steps + 1))
    N_trajectory = np.zeros((n_actions, number_of_steps + 1))

    # Variables to calculate arithmetic mean
    sum_rewards_per_action = np.zeros(n_actions)
    N_per_action = np.zeros(n_actions)

    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        # Uniformly random action selection
        action = int(rng.choice(n_actions))
        reward = env.step(action)

        # Update counts and sum of rewards for the chosen action
        N_per_action[action] += 1
        sum_rewards_per_action[action] += reward

        # Store counts for plotting
        N_trajectory[:, t+1] = N_per_action

        # Calculate Q(a) as arithmetic mean for all actions
        for i in range(n_actions):
            if N_per_action[i] > 0:
                Q_trajectory[i, t+1] = sum_rewards_per_action[i] / N_per_action[i]
            else:
                Q_trajectory[i, t+1] = 0.0 # Keep 0 if no rewards yet

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q_trajectory[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].set_title('Q-values over steps (Uniform Random)')
        axarr[0].grid()

        for i in range(n_actions):
            axarr[1].plot(N_trajectory[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].set_title('Action Counts over steps (Uniform Random)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q_trajectory[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q_trajectory, N=N_trajectory, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

以下に一様分布で行動を選択しつつづけたときの行動価値と行動選択回数のグラフを示します。

In [ ]:
_ = uniform_random_method(env=env, number_of_steps=1000)

## $\varepsilon$-greedy方策
$\varepsilon$-greedyは、価値ベースの強化学習において探索（exploration）と活用（exploitation）のバランスをとるための手法です。現在の行動価値関数の推定値が $Q(a)$ であるとき、行動 $a$ を選択する確率は以下のように与えられます。
\begin{equation}
  \pi (a) =
  \begin{cases}
    \epsilon/\lvert A \rvert + (1 - \epsilon)/|\mathrm{argmax}_a \,Q| & \mathrm{if} \; \;
       a \in \mathrm{arg}\max_a Q(a), \\
    \epsilon/\lvert A \rvert & \mathrm{otherwise},
  \end{cases}
\end{equation}
ここで $\varepsilon \in [0, 1]$ はハイパーパラメータであり、$\lvert A \rvert$ は選択可能な行動の数です。

$\mathrm{argmax}_a \,Q$ は $Q$ を最大にする行動の集合になります。たとえば 5 個の行動価値が $Q = [1.0, 0.0, 0.0, 1.0, 0.0]$ となっていれば $Q(a_0)$ と $Q(a_3)$ が最大の価値を持つので
\begin{equation}
  \mathrm{arg} \max_a Q(a) = \{ a_0, a_3 \}
\end{equation}
となります。そのとき $|\mathrm{argmax}_a \,Q| = 2$ となります。
\begin{equation}
  a \in \mathrm{arg} \max_a Q(a)
\end{equation}
と書かれたとき、$a$ は右辺の集合のなかのどれか一つを指します。今の例だと $a$ は $a_0$ か $a_3$ となります。

In [ ]:
def argmax_random_tie(x, rng=None):
    """Return an argmax of x, breaking ties uniformly at random."""
    if rng is None:
        rng = np.random.default_rng()
    x = np.asarray(x)
    max_val = np.max(x)
    best = np.flatnonzero(x == max_val)
    return int(rng.choice(best))


def epsilon_greedy_policy(Q, epsilon=0.1):
    """Epsilon-greedy policy (returns a probability vector over actions).

    With random tie-breaking among argmax actions.
    """
    n_actions = Q.shape[0]
    policy = epsilon / n_actions * np.ones(n_actions)
    max_val = np.max(Q)
    best = np.flatnonzero(Q == max_val)
    policy[best] += (1 - epsilon) / len(best)
    return policy


def plot_epsilon_greedy_policy(Q, epsilon=0.1):
    policy = epsilon_greedy_policy(Q, epsilon)
    action = np.arange(Q.shape[0])

    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].bar(action, Q)
    axarr[0].set_xlabel('action a')
    axarr[0].set_ylabel('action value Q(a)')

    axarr[1].bar(action, policy, label=r'$\epsilon$ = %3.1f' % epsilon)
    axarr[1].set_xlabel('action a')
    axarr[1].set_ylabel(r'policy $\pi$ (a)')

    plt.legend()
    plt.show()


以下では5個の行動に対して行動価値をそれぞれ
$$ Q = [2.5, -2.5, 2.9, 1.2, 0.5]$$
と設定したとき、$\varepsilon$ の値によって方策 $\pi(a)$ がどのように変化するかを表示しています。

In [ ]:
# epsilon = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_epsilon_greedy_policy, Q=fixed(Q), epsilon=(0, 1, 0.05));


In [ ]:
def naive_epsilon_greedy_policy(Q, epsilon=0.1):
    """Naive epsilon-greedy policy using simple np.argmax.

    In case of ties, np.argmax always returns the first (smallest) index.
    """
    n_actions = Q.shape[0]
    # 一様ランダムな確率を割り当て
    policy = np.ones(n_actions) * (epsilon / n_actions)

    # 活用（Exploitation）: 単純な argmax を使用
    best_action = np.argmax(Q)
    policy[best_action] += (1.0 - epsilon)

    return policy

def compare_policies(Q, epsilon=0.1):
    p_random = epsilon_greedy_policy(Q, epsilon)
    p_naive = naive_epsilon_greedy_policy(Q, epsilon)
    action = np.arange(Q.shape[0])

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    width = 0.35
    ax.bar(action - width/2, p_random, width, label='Random Tie-break')
    ax.bar(action + width/2, p_naive, width, label='Naive (Smallest Index)')

    ax.set_xlabel('Action a')
    ax.set_ylabel('Probability')
    ax.set_title(f'Policy Comparison (epsilon={epsilon})')
    ax.set_xticks(action)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

### タイブレークの影響の可視化
すべての $Q$ 値が $0$ である初期状態において、両者の挙動の違いを確認します。

In [ ]:
Q_initial = np.zeros(5)
print("Initial Q values:", Q_initial)
compare_policies(Q_initial, epsilon=0.1)

# 価値ベースの手法

エージェントは期待報酬を学習します。エージェントが行動 $a$ を選択し、スカラー報酬 $r$ を受け取ると、期待報酬の推定値は次のように更新されます。
\begin{equation*}
  Q(a) = Q(a) + \alpha (r - Q(a)),
\end{equation*}
ここで $\alpha$ は学習率です。以下では、各ステップでの $Q$ 推定値とアクションのカウントを記録するシングルランのルーチンを提供します。

In [ ]:
def value_based_method(env,
                       epsilon=0.1,
                       number_of_steps=1000,
                       initial_Q=0.0,
                       seed=None,
                       show_plot=True):
    """Single-run bandit experiment.

    Parameters
    ----------
    env : GaussianBandit instance
    epsilon : float
        Exploration rate for epsilon-greedy.
    number_of_steps : int
    initial_Q : float or ndarray
        Initial value of the action-value function.
    seed : int or None
        Random seed that controls action sampling (not the environment).
    show_plot : bool
    """
    # 行動選択のための乱数生成器（環境の報酬生成用とは別）
    rng = np.random.default_rng(seed)

    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        policy = epsilon_greedy_policy(Q[:, t], epsilon=epsilon)
        action = int(rng.choice(n_actions, p=policy))

        reward = env.step(action)

        N[:, t+1] = N[:, t]
        N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1]
        Q[action, t+1] = Q[action, t] + alpha * (reward - Q[action, t])

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].set_title('Q-values over steps')
        axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

## UCB (Upper Confidence Bound) 方策
UCB方策は、探索と活用のバランスを取るための別の手法です。この方策では、各アクションの選択は、現在の推定される報酬と、そのアクションがどれだけ不確実であるか（または、どれだけ試行回数が少ないか）に基づいて行われます。UCB方策は、特に初期の試行回数が少ないアクションを優先することで、効率的な探索を促します。
アクション $a$ を選択する際のUCB値は、以下の式で計算されます。
$$ \text{UCB}_t(a) = Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} $$
ここで、$Q_t(a)$ はアクション $a$ の推定期待報酬、$N_t(a)$ はアクション $a$ が選択された回数、$t$ は現在のステップ数、$c$ は探索の度合いを調整する正の定数です。
まだ一度も選択されていないアクションは、無限大のUCB値を持つとみなし、優先的に選択されます。

In [ ]:
def select_ucb_action(Q_values, N_values, t, c=2.0, rng=None):
    """UCB policy action selection.

    Parameters
    ----------
    Q_values : array_like
        Current estimated action values.
    N_values : array_like
        Counts of how many times each action has been selected.
    t : int
        Current time step (number of total actions taken so far).
    c : float
        Exploration parameter for UCB.
    rng : numpy.random.Generator or None
        Random number generator for tie-breaking.

    Returns
    -------
    int
        The chosen action.
    """
    if rng is None:
        rng = np.random.default_rng()
    n_actions = Q_values.shape[0]
    ucb_scores = np.zeros(n_actions)

    # Prioritize actions that haven't been tried yet
    # If N_values[action] is 0, then ln(t)/N_values[action] would be infinity.
    # We handle this by ensuring every action is tried at least once.
    untried_actions = np.where(N_values == 0)[0]
    if len(untried_actions) > 0:
        return rng.choice(untried_actions)

    # Calculate UCB scores for actions that have been tried
    for action in range(n_actions):
        # UCB formula: Q(a) + c * sqrt(ln(t) / N(a))
        # Note: t here refers to the total number of steps, which is used for ln(t)
        # N(a) refers to the count of action 'a'
        ucb_scores[action] = Q_values[action] + c * np.sqrt(np.log(t) / N_values[action])

    # Select action with the highest UCB score, breaking ties randomly
    return argmax_random_tie(ucb_scores, rng=rng)

In [ ]:
def ucb_method(env,
                   c=2.0, # UCB parameter
                   number_of_steps=1000,
                   initial_Q=0.0,
                   seed=None,
                   show_plot=True):
    """Single-run bandit experiment using the UCB policy.

    Parameters
    ----------
    env : GaussianBandit instance
    c : float
        Exploration parameter for UCB.
    number_of_steps : int
    initial_Q : float or ndarray
        Initial value of the action-value function.
    seed : int or None
        Random seed that controls action sampling (not the environment).
    show_plot : bool
    """
    rng = np.random.default_rng(seed)

    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        # Select action using UCB policy
        # t + 1 is used for log(t) to ensure t is never 0 for the log function
        action = select_ucb_action(Q[:, t], N[:, t], t + 1, c=c, rng=rng)

        reward = env.step(action)

        # Update counts and Q-values
        N[:, t+1] = N[:, t]
        N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1] # Using sample average for Q-value updates
        Q[action, t+1] = Q[action, t] + alpha * (reward - Q[action, t])

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].set_title(f'Q-values over steps (UCB c={c})')
        axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

In [ ]:
# Generate a random seed for each execution
import numpy as np
current_run_seed = np.random.randint(0, 1_000_000)
epsilon = 0.05 #@param {type:"slider", min:0, max:1, step:0.05}
_ = value_based_method(env=env, epsilon=epsilon, seed=current_run_seed)

## 複数回の実行による平均化と比較

1回だけの実行（シングルラン）はノイズが多く、最初の数回の試行が学習軌道を支配してしまいます。探索戦略を公平に評価するために、多くの独立したシードで平均をとり、以下の2つの標準的な曲線を確認します。

- **ステップごとの平均報酬**（アクションが平均してどの程度良いか）
- **最適アクション選択率**（最適な腕がどのくらいの頻度で選ばれているか）

また、以下の3つの設定を比較します。

1. $\varepsilon$-greedy ($\varepsilon=0.1$, $Q_0=0$) — 方策による探索
2. greedy ($\varepsilon=0$, $Q_0=10$) — **楽観的初期値**による探索
3. $\varepsilon=0.01$ (ベースライン)

In [ ]:
def run_experiment(env,
                   method_function=None,
                   n_runs=200,
                   number_of_steps=1000,
                   base_seed=0,
                   **kwargs):
    """Run `n_runs` independent runs on the SAME environment.
    """

    avg_reward_sum = None
    opt_action_sum = None
    last = None

    for k in range(n_runs):
        result = method_function(env=env,
                                    number_of_steps=number_of_steps,
                                    seed=base_seed + k + 10000, # 行動選択用
                                    show_plot=False,
                                    **kwargs)
        if avg_reward_sum is None:
            avg_reward_sum = np.zeros(number_of_steps)
            opt_action_sum = np.zeros(number_of_steps)

        avg_reward_sum += result['rewards']
        opt_action_sum += (result['actions'] == result['optimal_action']).astype(float)
        last = result

    avg_reward = avg_reward_sum / n_runs
    opt_rate   = opt_action_sum / n_runs
    return avg_reward, opt_rate, last['Q']

In [ ]:
import matplotlib.pyplot as plt

# Compare three strategies, averaged over n_runs seeds
n_runs = 200
T = 1000

configs = [
    dict(label=r'$\varepsilon$=0.1, $Q_0$=0',  method_function=value_based_method, epsilon=0.1,  initial_Q=0.0),
    dict(label=r'$\varepsilon$=0.01, $Q_0$=0', method_function=value_based_method, epsilon=0.01, initial_Q=0.0),
    dict(label=r'greedy, $Q_0$=10 (optimistic)', method_function=value_based_method, epsilon=0.0,  initial_Q=10.0),
    dict(label=r'UCB c=2', method_function=ucb_method, c=2.0, initial_Q=0.0)
]

results = []
for cfg in configs:
    current_cfg = cfg.copy()
    label = current_cfg.pop('label')
    method_func = current_cfg.pop('method_function')
    avg_r, opt_r, Q_last = run_experiment(env=env, method_function=method_func, n_runs=n_runs, number_of_steps=T, **current_cfg)
    results.append((label, avg_r, opt_r, Q_last))

fig, axarr = plt.subplots(1, 2, figsize=(13, 5))
for label, avg_r, opt_r, _ in results:
    axarr[0].plot(avg_r, label=label)
    axarr[1].plot(opt_r, label=label)
axarr[0].set_xlabel('steps'); axarr[0].set_ylabel('average reward'); axarr[0].grid(); axarr[0].legend()
axarr[1].set_xlabel('steps'); axarr[1].set_ylabel('optimal action rate'); axarr[1].grid(); axarr[1].legend()
plt.tight_layout(); plt.show()

# And the per-arm Q trajectory of the last run (epsilon=0.1 run)
# Find the result for epsilon=0.1, initial_Q=0.0
Q_last_epsilon_01 = None
for label, avg_r, opt_r, Q_last in results:
    if label == r'$\varepsilon$=0.1, $Q_0$=0': # Matching the label for epsilon=0.1
        Q_last_epsilon_01 = Q_last
        break

if Q_last_epsilon_01 is not None:
    fig = plt.figure(figsize=(8, 5))
    for i in range(Q_last_epsilon_01.shape[0]):
        plt.plot(Q_last_epsilon_01[i, :], label='$a%i$' % i)
    plt.xlabel('steps'); plt.ylabel(r'Q(a)  (last run of $\varepsilon$=0.1)')
    plt.grid(); plt.legend(); plt.show()
else:
    print("Could not find Q_last for epsilon=0.1, initial_Q=0.0 to plot.")